# Measuring how two variables relate (correlation & association)

_Metrics & Evaluation — measuring models, data & statistics_

**How strongly do two things move together — and which number to trust for which kind of relationship.**

Two columns are associated if knowing one tells you something about the other. "How much?" is a single number, usually scaled so that 0 = no relationship and ±1 = a perfect one (sign tells you the direction for the signed measures).

       The catch: there are many kinds of "move together". A straight-line trend, a curved-but-always-rising trend, a category-to-category link, an any-shape dependence. No single number captures all of them, so you have a small toolbox and pick by the data type and the shape you expect.

---

This notebook is a practice scaffold for the **Measuring how two variables relate (correlation & association)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_wine

data = load_wine(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — scipy / scikit-learn / statsmodels

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.datasets import load_wine
from sklearn.feature_selection import mutual_info_classif
from statsmodels.stats.outliers_influence import variance_inflation_factor

wine = load_wine(as_frame=True)
df = wine.frame                      # 13 continuous features + 'target' (3 classes)

# --- 1) Pearson (linear), Spearman & Kendall (monotonic/rank) ---
x = df["alcohol"].values
y = df["proline"].values
print("Pearson  r:", stats.pearsonr(x, y))      # straight-line strength
print("Spearman p:", stats.spearmanr(x, y))     # monotonic strength (on ranks)
print("Kendall  t:", stats.kendalltau(x, y))    # concordant vs discordant pairs

# --- 2) point-biserial: a binary flag vs a continuous column ---
is_class0 = (df["target"] == 0).astype(int).values
print("point-biserial:", stats.pointbiserialr(is_class0, df["flavanoids"].values))

# --- 3) Mutual Information of every feature with the target (nonlinear-aware) ---
X = wine.data.values
mi = mutual_info_classif(X, wine.target.values, random_state=0)
mi = pd.Series(mi, index=wine.feature_names).sort_values(ascending=False)
print("\nMutual information with target:\n", mi.round(3).head(6))

# --- 4) Cramer's V for two CATEGORICAL columns (bin features into categories) ---
def cramers_v(a, b):
    table = pd.crosstab(a, b)
    chi2 = stats.chi2_contingency(table)[0]
    n = table.values.sum()
    k, m = table.shape
    return np.sqrt(chi2 / (n * (min(k, m) - 1)))      # in [0, 1]

alc_bin = pd.qcut(df["alcohol"], 3, labels=["low", "mid", "high"])
print("\nCramer's V (alcohol-bin vs wine class):",
      round(cramers_v(alc_bin, df["target"]), 3))

# --- 5) Variance Inflation Factor: diagnose multicollinearity before regression ---
feats = df[["alcohol", "flavanoids", "color_intensity", "hue",
            "od280/od315_of_diluted_wines", "proline"]].copy()
feats["const"] = 1.0                                  # statsmodels needs an intercept
vif = pd.Series(
    [variance_inflation_factor(feats.values, i) for i in range(feats.shape[1] - 1)],
    index=feats.columns[:-1])
print("\nVIF (>5-10 => multicollinear):\n", vif.round(2))

# scikit-learn / scipy also offer: distance correlation via dcor,
# MIC via the 'minepy' package, and partial correlation via 'pingouin'.

## Visualize the data & results

_Among several wine-chemistry features, which pairs move together — and is the trend linear (Pearson) or just monotonic (the rank story)?_

In [ ]:
import numpy as np
from sklearn.datasets import load_wine
from sklearn.feature_selection import mutual_info_classif

wine = load_wine()
names = wine.feature_names
pick = ["alcohol", "flavanoids", "color_intensity", "hue",
        "od280/od315_of_diluted_wines", "proline"]
idx = [names.index(p) for p in pick]

# --- heatmap: Pearson correlation matrix among the chosen features ---
X = wine.data[:, idx]
corr = np.corrcoef(X.T)                       # 6x6, diagonal = 1.0
print(np.round(corr, 2))

# --- bars: mutual information of EVERY feature with the 3-class target ---
mi = mutual_info_classif(wine.data, wine.target, random_state=0)
order = np.argsort(mi)[::-1][:8]
for i in order:
    print(f"{names[i]:>30s}  MI = {mi[i]:.3f}")

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You plot feature X against the target Y and the scatter is a clear smile-shaped (U) curve, but scipy.stats.pearsonr returns r ≈ 0.02. A teammate concludes "X is useless, drop it." Are they right?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall what Pearson r measures. — _r is the strength of a straight-line trend only. A symmetric U-shape has no net linear slope, so r ≈ 0 even though the relationship is strong._
- Reach for a measure that sees curves. — _Distance correlation or Mutual Information (MI) detect any-shape dependence; Spearman catches monotonic ones (but a U is not monotonic, so even Spearman would be low here)._
- Confirm visually. — _The scatter already shows the structure — this is exactly the Anscombe's-quartet lesson: never trust r without the picture._

**Answer:** No. Pearson $r\approx 0$ only rules out a straight-line trend, not a relationship. The U-shape is a real (nonlinear) dependence that $r$ is structurally blind to. Keep the feature: confirm with distance correlation or Mutual Information (both pick up the U), and consider a nonlinear model or a squared term. Spearman would also be low here because a U is not monotonic, which is itself the clue that the link is curved, not absent.

</details>

**Problem 2.** In a regression you find that two predictors, "height in cm" and "height in inches" (someone added both), each have a sensible correlation with the target, but the fitted coefficients are huge, opposite in sign, and flip when you refit. Which diagnostic catches this and what's the fix?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Name the problem. — _The two columns are near-perfect linear copies of each other — extreme multicollinearity. Pairwise correlation between them is ~1._
- Compute VIF. — _The Variance Inflation Factor for each is $1/(1-R_j^2)$ with $R_j^2\approx 1$, so VIF explodes (→ very large), flagging the redundancy that pairwise correlation alone might understate across many features._
- Remove the redundancy. — _Drop one column (or combine them), or use ridge regression, to stabilize the coefficients._

**Answer:** VIF (Variance Inflation Factor) catches it: with one feature almost perfectly predicted by the other, $R_j^2\to 1$ and $\text{VIF}=1/(1-R_j^2)\to\infty$ (well past the ~5–10 warning line). High multicollinearity inflates the coefficient standard errors, which is why the estimates are huge, sign-flipping, and unstable. Fix: drop one of the duplicate columns (or merge them), or apply ridge regularization. VIF is the right tool because it measures a feature against all the others jointly, not just one pair at a time.

</details>

**Problem 3.** A report claims "cities with more firefighters have more fire damage, so firefighters cause damage." Both columns are continuous and Pearson r = 0.8. What measure would you compute to challenge the causal claim, and what is the likely real story?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Separate correlation from causation. — _A strong $r$ only says the two move together; it says nothing about cause. A lurking third variable (confounder) can drive both._
- Identify the confounder. — _City size drives both: bigger cities have more firefighters AND more (and bigger) fires._
- Use partial correlation. — _Partial correlation of firefighters and damage controlling for city size removes the confounder's effect from both; if the link collapses toward 0, firefighters weren't the driver._

**Answer:** Compute the partial correlation between firefighters and damage while controlling for city size (regress city size out of both columns, then correlate the residuals). The original $r=0.8$ is almost certainly confounded: larger cities have both more firefighters and more fire damage. After removing city size, the partial correlation should drop sharply toward 0, showing the firefighter–damage link was spurious. This is the textbook "correlation is not causation" trap, and partial correlation is the tool that exposes the lurking variable.

</details>